# 14. Comparacion temporal de determinantes del ingreso laboral individual

**Pregunta.** ¿Como cambian entre 2018, 2020, 2022 y 2024 las asociaciones entre caracteristicas personales, laborales y territoriales y el ingreso laboral o de negocio trimestral de una persona adulta con ingreso positivo?

Cada fila es una persona; los cuatro levantamientos ENIGH son **cortes transversales repetidos**, no un panel. No se sigue a las mismas personas ni hogares. Los modelos separados describen cada corte; el modelo pooled compara promedios condicionados y las interacciones con año permiten probar diferencias de pendientes. Una variable significativa en un año y no en otro no prueba por si sola un cambio. Las asociaciones no son efectos causales.

La geografia principal son las regiones Banxico; las entidades federativas son una sensibilidad separada. La inferencia usa toda la muestra valida por año, sin seleccion por significancia ni holdout. El notebook 13 sigue siendo el diagnostico detallado local de 2024. Este notebook solo lee cuatro bases individuales ya exportadas por el 12 y no usa bases de hogares.

In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import statsmodels.api as sm
from IPython.display import display
from scipy.stats import norm

ANIOS = [2018, 2020, 2022, 2024]
ANIO_REFERENCIA = 2024
ESPECIFICACION_GEOGRAFICA_PRINCIPAL = "region_banxico"
EJECUTAR_ESPECIFICACION_ENTIDAD = True
MOSTRAR_SUMMARIES_COMPLETOS = True
EXPORTAR_RESULTADOS = True
EJECUTAR_MODELOS_PONDERADOS = True
EJECUTAR_INTERACCIONES_COMPLETAS = True

# Solo se usan si EJECUTAR_INTERACCIONES_COMPLETAS es False.
BLOQUES_INTERACCION = ["individuales", "laborales", "localidad", "region"]
VARIABLES_INTERACCION_MANUAL = []
EJECUTAR_ESPECIFICACION_AMPLIADA = False
ALFA = 0.05
EDAD_MINIMA = 18

if ANIOS != [2018, 2020, 2022, 2024] or ANIO_REFERENCIA not in ANIOS:
    raise ValueError("Revisar años y referencia temporal.")
if ESPECIFICACION_GEOGRAFICA_PRINCIPAL != "region_banxico":
    raise ValueError("El contrato fija region_banxico como especificacion principal.")
if not 0 < ALFA < 1:
    raise ValueError("ALFA debe estar entre cero y uno.")

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "README.md").exists():
    raise FileNotFoundError("Abrir Jupyter desde la raiz del proyecto o notebooks/.")

KEY_COLS = ["anio", "folioviv", "foliohog", "numren"]
HOGAR_COLS = KEY_COLS[:3]
TARGET_NOMINAL = "ingreso_persona_laboral_negocio_tri"
LOG_NOMINAL = "log1p_ingreso_persona_laboral_negocio_tri"
variables_comunes_numericas = ["edad", "n_trabajos", "horas_trabajos_total"]
variables_comunes_categoricas = [
    "sexo_desc", "hablaind_desc", "segsoc_desc", "subor_principal_desc",
    "contrato_principal_desc", "tam_loc_desc",
]
variable_geografica_regional = ["region_banxico"]
variable_geografica_estatal = ["entidad"]
variables_pendientes = [
    "nivelaprob_desc", "parentesco_desc", "tam_emp_principal_desc", "est_socio_desc",
]
ESTRUCTURALES = {
    "subor_principal_desc": "No aplica: sin trabajo principal reportado",
    "contrato_principal_desc": "No aplica: sin contrato principal documentado",
}
REFERENCIAS = {
    "sexo_desc": "Hombre",
    "hablaind_desc": "No",
    "segsoc_desc": "No",
    "subor_principal_desc": "No",
    "contrato_principal_desc": "No",
    "tam_loc_desc": "Localidades con 100 000 y más habitantes",
    "region_banxico": "Centro",
    "entidad": "Ciudad de Mexico",
    "nivelaprob_desc": "Ninguno",
}
BLOQUES = {
    "individuales": ["edad", "sexo_desc", "hablaind_desc"],
    "laborales": ["n_trabajos", "horas_trabajos_total", "segsoc_desc",
                  "subor_principal_desc", "contrato_principal_desc"],
    "localidad": ["tam_loc_desc"],
    "region": ["region_banxico"],
    "escolaridad": ["nivelaprob_desc"],
}
INPUTS = {
    anio: PROJECT_ROOT / "data" / "processed" / f"determinantes_{anio}"
          / f"base_interpretable_personas_{anio}.csv.gz"
    for anio in ANIOS
}
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "comparacion_temporal_determinantes"
display(pd.DataFrame([
    {"parametro": k, "valor": str(v)} for k, v in {
        "ANIOS": ANIOS, "ANIO_REFERENCIA": ANIO_REFERENCIA,
        "ESPECIFICACION_GEOGRAFICA_PRINCIPAL": ESPECIFICACION_GEOGRAFICA_PRINCIPAL,
        "EJECUTAR_ESPECIFICACION_ENTIDAD": EJECUTAR_ESPECIFICACION_ENTIDAD,
        "MOSTRAR_SUMMARIES_COMPLETOS": MOSTRAR_SUMMARIES_COMPLETOS,
        "EXPORTAR_RESULTADOS": EXPORTAR_RESULTADOS,
        "EJECUTAR_MODELOS_PONDERADOS": EJECUTAR_MODELOS_PONDERADOS,
        "EJECUTAR_INTERACCIONES_COMPLETAS": EJECUTAR_INTERACCIONES_COMPLETAS,
        "EJECUTAR_ESPECIFICACION_AMPLIADA": EJECUTAR_ESPECIFICACION_AMPLIADA,
        "ALFA": ALFA, "EDAD_MINIMA": EDAD_MINIMA,
    }.items()
]))

## 1. Lectura y auditoria de las cuatro bases

Se validan archivos, llaves, año, edad, ingreso positivo, transformaciones, geografia y variables de diseño. Los conteos mostrados son agregados: no se imprimen folios. Los targets reales tienen sufijo del **año de origen** en el archivo, aunque el factor los expresa aproximadamente en pesos de 2024. La consistencia aritmetica no valida por si misma la pertinencia economica del factor.

In [ ]:
def nombres_targets(anio):
    return {
        "nivel_nominal": TARGET_NOMINAL,
        "log_nominal": LOG_NOMINAL,
        "nivel_real_2024": f"ingreso_persona_laboral_negocio_tri_real_{anio}",
        "log_real_2024": f"log1p_ingreso_persona_laboral_negocio_tri_real_{anio}",
    }


def auditar_base(anio, ruta):
    if not ruta.exists():
        raise FileNotFoundError(ruta)
    base = pd.read_csv(ruta, low_memory=False)
    targets = nombres_targets(anio)
    obligatorias = (
        KEY_COLS + list(targets.values()) + variables_comunes_numericas
        + variables_comunes_categoricas + variable_geografica_regional
        + variable_geografica_estatal + variables_pendientes
        + ["factor", "est_dis", "upm", "cve_ent", "deflactor_2024",
           "segsoc_desc_original"]
    )
    ausentes = sorted(set(obligatorias) - set(base.columns))
    if ausentes:
        raise KeyError(f"{anio}: columnas ausentes: {ausentes}")
    nominal = pd.to_numeric(base[TARGET_NOMINAL], errors="coerce")
    real = pd.to_numeric(base[targets["nivel_real_2024"]], errors="coerce")
    log_nominal = pd.to_numeric(base[LOG_NOMINAL], errors="coerce")
    log_real = pd.to_numeric(base[targets["log_real_2024"]], errors="coerce")
    deflactor = pd.to_numeric(base["deflactor_2024"], errors="coerce")
    edad = pd.to_numeric(base["edad"], errors="coerce")
    if base.empty or base[KEY_COLS].isna().any().any():
        raise ValueError(f"{anio}: base vacia o llaves incompletas.")
    if not base["anio"].eq(anio).all() or base.duplicated(KEY_COLS).any():
        raise ValueError(f"{anio}: año incorrecto o llave personal duplicada.")
    if not edad.ge(EDAD_MINIMA).all() or not (nominal.gt(0) & np.isfinite(nominal)).all():
        raise ValueError(f"{anio}: edad/ingreso fuera del universo.")
    if not (real.gt(0) & np.isfinite(real)).all() or not (
        deflactor.gt(0) & np.isfinite(deflactor)
    ).all() or deflactor.nunique() != 1:
        raise ValueError(f"{anio}: target real o factor invalido.")
    if anio == ANIO_REFERENCIA and not np.isclose(deflactor.iloc[0], 1):
        raise ValueError("El factor del año de referencia debe ser uno.")
    if not (
        np.allclose(log_nominal, np.log1p(nominal), rtol=1e-10, atol=1e-10)
        and np.allclose(real, nominal * deflactor, rtol=1e-10, atol=1e-10)
        and np.allclose(log_real, np.log1p(real), rtol=1e-10, atol=1e-10)
    ):
        raise ValueError(f"{anio}: targets o transformaciones inconsistentes.")
    if base[["region_banxico", "entidad", "cve_ent"]].isna().any().any():
        raise ValueError(f"{anio}: geografia incompleta.")
    codigos = base[["cve_ent", "entidad"]].drop_duplicates()
    if (
        codigos.cve_ent.nunique() != codigos.entidad.nunique()
        or codigos.duplicated("cve_ent").any()
        or codigos.duplicated("entidad").any()
    ):
        raise ValueError(f"{anio}: codigo y nombre estatal no son uno a uno.")
    peso = pd.to_numeric(base["factor"], errors="coerce")
    diseno_valido = bool(
        peso.gt(0).all() and np.isfinite(peso).all()
        and base[["est_dis", "upm"]].notna().all().all()
        and base["est_dis"].nunique() > 1 and base["upm"].nunique() > 1
    )
    fila = {
        "anio": anio, "archivo": str(ruta.relative_to(PROJECT_ROOT)),
        "n_personas": len(base),
        "n_hogares": base[HOGAR_COLS].drop_duplicates().shape[0],
        "duplicados_persona": int(base.duplicated(KEY_COLS).sum()),
        "edad_minima_observada": float(edad.min()),
        "ingreso_nominal_minimo": float(nominal.min()),
        "deflactor_2024": float(deflactor.iloc[0]),
        "regiones": base.region_banxico.nunique(),
        "entidades": base.entidad.nunique(),
        "factor_valido_y_diseno_disponible": diseno_valido,
        "faltantes_factor": int(peso.isna().sum()),
        "faltantes_est_dis": int(base.est_dis.isna().sum()),
        "faltantes_upm": int(base.upm.isna().sum()),
    }
    return base, fila


bases = {}
filas_auditoria = []
for anio in ANIOS:
    bases[anio], fila = auditar_base(anio, INPUTS[anio])
    filas_auditoria.append(fila)
codigos_estatales = pd.concat([
    base[["cve_ent", "entidad"]].drop_duplicates() for base in bases.values()
], ignore_index=True).drop_duplicates()
if (codigos_estatales.groupby("cve_ent").entidad.nunique().gt(1).any()
    or codigos_estatales.groupby("entidad").cve_ent.nunique().gt(1).any()):
    raise ValueError("Codigo/nombre estatal cambia entre años.")
auditoria_bases_temporales = pd.DataFrame(filas_auditoria)
display(auditoria_bases_temporales)

## 2. Contrato comun, faltantes y referencias

Nucleo: tres predictores numericos y seis categoricos; la region se añade por separado. Los faltantes de subordinacion y contrato corresponden a rutas estructurales del cuestionario y reciben exactamente las etiquetas usadas en el notebook 13. Los faltantes originales permanecen en la auditoria. No se reagrupan categorias raras. Las referencias se fijan una vez; si falta una, se muestra la interseccion candidata y se detiene el ajuste. La codificacion de seguridad social se verifica contra el original conservado por el 12.

Escolaridad no entra en el nucleo. La tabla muestra categorias y frecuencias por año; aun si coinciden las cadenas, una ampliacion exige aprobacion semantica de codigos. Parentesco queda como sensibilidad futura. Tamaño de empresa y estrato socioeconomico requieren revisar dependencia laboral y nivel conceptual, respectivamente.

In [ ]:
SEGSOC_ORIGEN_ESPERADO = {
    2018: {"No": "No", "Sí": "Sí"},
    2020: {"No nóicpircseD .0202 y": "No", "Sí": "Sí"},
    2022: {"No nóicpircseD .2202 y": "No", "Sí": "Sí"},
    2024: {"No": "No", "Sí": "Sí"},
}
bases_modelo = {}
for anio, base in bases.items():
    pares = base[["segsoc_desc_original", "segsoc_desc"]].drop_duplicates()
    observados = dict(zip(pares.segsoc_desc_original, pares.segsoc_desc))
    if observados != SEGSOC_ORIGEN_ESPERADO[anio]:
        raise ValueError(f"{anio}: etiquetas de segsoc_desc inesperadas: {observados}")
    preparada = base.copy(deep=False)
    for variable, etiqueta in ESTRUCTURALES.items():
        preparada[variable] = preparada[variable].fillna(etiqueta)
    bases_modelo[anio] = preparada

variables_comparar = (
    variables_comunes_numericas + variables_comunes_categoricas
    + variable_geografica_regional + variable_geografica_estatal
    + variables_pendientes
)
filas_comparabilidad = []
for variable in variables_comparar:
    numerica = variable in variables_comunes_numericas
    geografica = variable in variable_geografica_regional + variable_geografica_estatal
    pendiente = variable in variables_pendientes
    fila = {
        "variable": variable,
        "tipo": "numerica" if numerica else "categorica",
        "nivel_conceptual": (
            "individual" if variable in BLOQUES["individuales"]
            or variable == "nivelaprob_desc" else
            "laboral" if variable in BLOQUES["laborales"]
            or variable == "tam_emp_principal_desc" else
            "territorial" if geografica or variable == "tam_loc_desc" else
            "contexto_hogar" if variable == "est_socio_desc" else "parentesco"
        ),
        "referencia_comun": REFERENCIAS.get(variable),
        "incluida_nucleo": not pendiente,
        "incluida_ampliada": False,
        "pendiente": pendiente,
        "regla_armonizacion": (
            "2020/2022: correccion exacta de segsoc_desc validada con original"
            if variable == "segsoc_desc" else
            "faltante estructural con etiqueta explicita"
            if variable in ESTRUCTURALES else
            "sin recodificacion; exige homologacion semantica"
            if pendiente else "misma definicion provisional"
        ),
        "motivo": (
            "etiquetas 2024 distintas; codigos no exportados, equivalencia pendiente"
            if variable == "nivelaprob_desc" else
            "etiquetas y categorias difieren; sensibilidad futura"
            if variable == "parentesco_desc" else
            "dependencia con subordinacion y faltante estructural"
            if variable == "tam_emp_principal_desc" else
            "estrato del hogar; revisar nivel conceptual"
            if variable == "est_socio_desc" else ""
        ),
    }
    for anio in ANIOS:
        base = bases[anio]
        preparada = bases_modelo[anio]
        disponible = variable in base
        fila[f"disponible_{anio}"] = disponible
        fila[f"faltantes_{anio}"] = int(base[variable].isna().sum()) if disponible else None
        fila[f"faltantes_pct_{anio}"] = float(base[variable].isna().mean() * 100) if disponible else None
        if disponible and not numerica:
            conteos = preparada[variable].value_counts(dropna=False)
            fila[f"categorias_{anio}"] = " | ".join(sorted(map(str, conteos.index)))
            fila[f"frecuencia_minima_{anio}"] = int(conteos.min())
        else:
            fila[f"categorias_{anio}"] = None
            fila[f"frecuencia_minima_{anio}"] = None
    filas_comparabilidad.append(fila)

comparabilidad_variables = pd.DataFrame(filas_comparabilidad)
display(comparabilidad_variables)
display(comparabilidad_variables.loc[
    comparabilidad_variables.variable.isin(variables_pendientes),
    ["variable", "referencia_comun", "motivo"]
    + [f"categorias_{anio}" for anio in ANIOS]
])

In [ ]:
CATEGORIAS_COMUNES = {}
filas_referencias = []
for variable in variables_comunes_categoricas + ["region_banxico", "entidad"]:
    conjuntos = {
        anio: set(bases_modelo[anio][variable].dropna().astype(str))
        for anio in ANIOS
    }
    referencia = REFERENCIAS[variable]
    interseccion = set.intersection(*conjuntos.values())
    iguales = all(c == conjuntos[ANIO_REFERENCIA] for c in conjuntos.values())
    for anio in ANIOS:
        filas_referencias.append({
            "anio": anio, "variable": variable, "referencia": referencia,
            "referencia_presente": referencia in conjuntos[anio],
            "categorias": " | ".join(sorted(conjuntos[anio])),
            "ausentes_vs_2024": " | ".join(sorted(conjuntos[ANIO_REFERENCIA] - conjuntos[anio])),
            "nuevas_vs_2024": " | ".join(sorted(conjuntos[anio] - conjuntos[ANIO_REFERENCIA])),
            "categorias_iguales_cuatro_anios": iguales,
        })
    if not iguales or referencia not in interseccion:
        print(pd.DataFrame(filas_referencias).loc[lambda d: d.variable.eq(variable)])
        raise ValueError(
            f"{variable}: categorias/referencia incompatibles. "
            f"Referencias comunes candidatas (no seleccionadas): {sorted(interseccion)}"
        )
    CATEGORIAS_COMUNES[variable] = [referencia] + sorted(conjuntos[ANIO_REFERENCIA] - {referencia})
referencias_categoricas = pd.DataFrame(filas_referencias)
display(referencias_categoricas[["anio", "variable", "referencia",
                                "referencia_presente", "categorias_iguales_cuatro_anios"]])
escolaridad_igual = all(
    set(bases_modelo[anio].nivelaprob_desc.dropna())
    == set(bases_modelo[ANIO_REFERENCIA].nivelaprob_desc.dropna())
    for anio in ANIOS
)
if EJECUTAR_ESPECIFICACION_AMPLIADA:
    raise ValueError(
        "Escolaridad ampliada detenida: las etiquetas 2024 difieren y no hay "
        "homologacion semantica aprobada de codigos en las bases individuales."
    )
estado_escolaridad = "pendiente_homologacion" if not escolaridad_igual else "pendiente_aprobacion_semantica"
display(pd.DataFrame([{"variable": "nivelaprob_desc", "estado": estado_escolaridad,
                       "categorias_iguales": escolaridad_igual,
                       "especificacion_ampliada_ejecutada": False}]))

## 3. Targets, diseño y propositos

Se conservan cuatro escalas: nivel nominal trimestral, log1p nominal, nivel real aproximado en pesos de 2024 y log1p real. El analisis temporal **principal** usa log1p real; en 2024 coincide exactamente con log1p nominal porque el factor vale uno. El antecedente de 2024 encontro mejor ajuste con el target logaritmico. Los coeficientes de log1p ayudan a leer diferencias relativas: para una dummy frente a su referencia, 100(exp(beta)-1) es la diferencia porcentual exacta en 1+ingreso; 100*beta es una aproximacion para beta pequeño. Para edad, horas u otro predictor continuo, la formula corresponde a un cambio de una unidad, condicionando lo demas.

El factor real actual fue calibrado con un benchmark de ingreso corriente **del hogar** y aplicado aqui al ingreso laboral **individual**. Es una aproximacion anual que requiere aprobacion metodologica antes de interpretar tendencias reales. Los interceptos y betas en pesos **nominales** no se comparan directamente entre años como poder adquisitivo. El modelo nominal en niveles se deja como continuidad secundaria.

**Inferencia:** todas las personas validas, sin particion, para coeficientes, intervalos y Wald. **Prediccion:** una eventual evaluacion separada usaria train/valid agrupados por hogar y preprocesamiento aprendido solo en train; MAE/RMSE no deciden significancia. Este notebook no mezcla ni exporta coeficientes de train como si fueran inferencia final.

In [ ]:
ESPECIFICACIONES = {
    "regional_nucleo": variables_comunes_numericas + variables_comunes_categoricas
                      + variable_geografica_regional,
}
if EJECUTAR_ESPECIFICACION_ENTIDAD:
    ESPECIFICACIONES["estatal_sensibilidad"] = (
        variables_comunes_numericas + variables_comunes_categoricas
        + variable_geografica_estatal
    )
for nombre, variables in ESPECIFICACIONES.items():
    if "region_banxico" in variables and "entidad" in variables:
        raise ValueError(f"{nombre}: no combinar dos geografias.")
    if len(variables) != len(set(variables)):
        raise ValueError(f"{nombre}: predictores repetidos.")

TARGETS_ANUALES = ["log_real_2024", "nivel_nominal"]
display(pd.DataFrame([
    {"especificacion": nombre, "predictores": " | ".join(variables),
     "target_principal": "log_real_2024", "target_secundario": "nivel_nominal"}
    for nombre, variables in ESPECIFICACIONES.items()
]))


def construir_matriz(base, variables):
    matriz = pd.DataFrame(index=base.index)
    info = [{"termino": "const", "variable": "intercepto",
             "categoria": None, "referencia": None, "tipo": "intercepto"}]
    for variable in variables:
        if variable in variables_comunes_numericas:
            serie = pd.to_numeric(base[variable], errors="coerce")
            if serie.isna().any() or not np.isfinite(serie).all():
                raise ValueError(f"{variable}: numerica no finita o faltante.")
            matriz[variable] = serie.astype(float)
            info.append({"termino": variable, "variable": variable,
                         "categoria": None, "referencia": None, "tipo": "numerica"})
        else:
            if variable not in CATEGORIAS_COMUNES:
                raise KeyError(f"Categorias no aprobadas para {variable}.")
            serie = base[variable].astype(str)
            if set(serie) != set(CATEGORIAS_COMUNES[variable]):
                raise ValueError(f"{variable}: categorias observadas distintas del contrato.")
            for categoria in CATEGORIAS_COMUNES[variable][1:]:
                termino = f"{variable}__{categoria}"
                matriz[termino] = serie.eq(categoria).astype(float)
                info.append({"termino": termino, "variable": variable,
                             "categoria": categoria,
                             "referencia": REFERENCIAS[variable],
                             "tipo": "categorica"})
    matriz = sm.add_constant(matriz, has_constant="add")
    if matriz.isna().any().any() or not np.isfinite(matriz.to_numpy()).all():
        raise ValueError("Matriz incompleta o no finita.")
    if np.linalg.matrix_rank(matriz.to_numpy().T @ matriz.to_numpy()) != matriz.shape[1]:
        raise ValueError("Matriz sin rango completo; revisar dependencia exacta.")
    return matriz, pd.DataFrame(info)


def codigos_hogar(base):
    codigos, _ = pd.factorize(pd.MultiIndex.from_frame(base[HOGAR_COLS]))
    if len(np.unique(codigos)) < 2:
        raise ValueError("Se requieren al menos dos hogares para EE agrupados.")
    return codigos


def variantes_covarianza(ajuste, base):
    return {
        "convencional": ajuste,
        "HC3": ajuste.get_robustcov_results(cov_type="HC3", use_t=False),
        "cluster_hogar": ajuste.get_robustcov_results(
            cov_type="cluster", groups=codigos_hogar(base),
            use_correction=True, use_t=False
        ),
    }


def coeficientes_de(modelo, info, contexto):
    meta = info.set_index("termino")
    terminos = modelo.model.exog_names
    intervalos = np.asarray(modelo.conf_int(alpha=ALFA))
    filas = []
    for j, termino in enumerate(terminos):
        detalle = meta.loc[termino]
        filas.append({
            **contexto, "termino": termino, "variable": detalle["variable"],
            "categoria": detalle["categoria"], "referencia": detalle["referencia"],
            "tipo_variable": detalle["tipo"],
            "coeficiente": float(np.asarray(modelo.params)[j]),
            "error_estandar": float(np.asarray(modelo.bse)[j]),
            "estadistico": float(np.asarray(modelo.tvalues)[j]),
            "p_value": float(np.asarray(modelo.pvalues)[j]),
            "intervalo_inferior": float(intervalos[j, 0]),
            "intervalo_superior": float(intervalos[j, 1]),
            "n_personas": int(modelo.nobs),
            "r2": float(modelo.rsquared),
            "r2_ajustado": float(modelo.rsquared_adj),
            "aic": float(modelo.aic), "bic": float(modelo.bic),
            "categoria_ausente": False, "modelo_no_estimable": False,
        })
    return pd.DataFrame(filas)


def pruebas_categoricas(modelo, info, contexto):
    filas = []
    nombres = modelo.model.exog_names
    for variable, bloque in info.loc[info.tipo.eq("categorica")].groupby("variable"):
        terminos = bloque.termino.tolist()
        restriccion = np.zeros((len(terminos), len(nombres)))
        for j, termino in enumerate(terminos):
            restriccion[j, nombres.index(termino)] = 1
        prueba = modelo.f_test(restriccion)
        filas.append({
            **contexto, "variable": variable, "terminos_probados": " | ".join(terminos),
            "estadistico_F": float(prueba.fvalue),
            "gl_numerador": len(terminos),
            "gl_denominador": float(prueba.df_denom),
            "p_value": float(prueba.pvalue),
        })
    return pd.DataFrame(filas)

## 4. Modelos separados por año

Cada año recibe exactamente los mismos predictores, transformaciones y referencias. OLS log real es principal; el nivel nominal es secundario. HC3 y errores agrupados por hogar son tres lecturas de incertidumbre del **mismo ajuste**, no tres selecciones de variables. Una prueba F conjunta pregunta si todas las dummies de una variable categorica son cero simultaneamente; puede diferir del p-value de una sola categoria. Un IC de 95% muestra valores compatibles con el modelo, no causalidad.

La sensibilidad estatal reemplaza region; no se introducen ambas geografias en una regresion. La WLS usa factor normalizado y se ejecuta solo si factor, estrato y UPM pasan la auditoria. WLS y agrupacion por hogar **no** reconstruyen el diseño complejo completo.

In [ ]:
filas_metricas_anuales = []
tablas_coeficientes_anuales = []
tablas_pruebas_anuales = []
DISENO_VALIDO_CUATRO = bool(
    auditoria_bases_temporales.factor_valido_y_diseno_disponible.all()
)
PONDERADOS_EJECUTADOS = EJECUTAR_MODELOS_PONDERADOS and DISENO_VALIDO_CUATRO

for anio in ANIOS:
    base = bases_modelo[anio]
    targets = nombres_targets(anio)
    n_hogares = base[HOGAR_COLS].drop_duplicates().shape[0]
    for especificacion, variables in ESPECIFICACIONES.items():
        X, info = construir_matriz(base, variables)
        for escala_target in TARGETS_ANUALES:
            y = pd.to_numeric(base[targets[escala_target]], errors="coerce")
            if y.isna().any() or not np.isfinite(y).all():
                raise ValueError(f"{anio}: target {escala_target} no finito.")
            ajuste = sm.OLS(y, X).fit()
            if MOSTRAR_SUMMARIES_COMPLETOS and (
                especificacion == "regional_nucleo"
                and escala_target == "log_real_2024"
            ):
                print(f"OLS principal {anio}: {especificacion}, {escala_target}")
                display(ajuste.summary())
            contexto = {
                "anio": anio, "especificacion": especificacion,
                "escala_target": escala_target, "estimador": "OLS",
                "n_hogares": n_hogares,
            }
            filas_metricas_anuales.append({
                **contexto, "n_personas": int(ajuste.nobs),
                "r2": float(ajuste.rsquared),
                "r2_ajustado": float(ajuste.rsquared_adj),
                "aic": float(ajuste.aic), "bic": float(ajuste.bic),
            })
            for covarianza, resultado in variantes_covarianza(ajuste, base).items():
                etiqueta = {**contexto, "tipo_covarianza": covarianza}
                tablas_coeficientes_anuales.append(
                    coeficientes_de(resultado, info, etiqueta)
                )
                tablas_pruebas_anuales.append(
                    pruebas_categoricas(resultado, info, etiqueta)
                )
        if (
            PONDERADOS_EJECUTADOS and especificacion == "regional_nucleo"
        ):
            pesos = pd.to_numeric(base["factor"], errors="coerce")
            pesos_normalizados = pesos / pesos.mean()
            y = pd.to_numeric(base[targets["log_real_2024"]], errors="coerce")
            ajuste_wls = sm.WLS(y, X, weights=pesos_normalizados).fit()
            contexto = {
                "anio": anio, "especificacion": especificacion,
                "escala_target": "log_real_2024", "estimador": "WLS_sensibilidad",
                "n_hogares": n_hogares,
            }
            filas_metricas_anuales.append({
                **contexto, "n_personas": int(ajuste_wls.nobs),
                "r2": float(ajuste_wls.rsquared),
                "r2_ajustado": float(ajuste_wls.rsquared_adj),
                "aic": float(ajuste_wls.aic), "bic": float(ajuste_wls.bic),
            })
            for covarianza, resultado in variantes_covarianza(ajuste_wls, base).items():
                etiqueta = {**contexto, "tipo_covarianza": covarianza}
                tablas_coeficientes_anuales.append(
                    coeficientes_de(resultado, info, etiqueta)
                )
                tablas_pruebas_anuales.append(
                    pruebas_categoricas(resultado, info, etiqueta)
                )

metricas_modelos_anuales = pd.DataFrame(filas_metricas_anuales)
coeficientes_modelos_anuales = pd.concat(tablas_coeficientes_anuales, ignore_index=True)
pruebas_conjuntas_anuales = pd.concat(tablas_pruebas_anuales, ignore_index=True)
display(metricas_modelos_anuales)
display(pruebas_conjuntas_anuales.loc[
    pruebas_conjuntas_anuales.especificacion.eq("regional_nucleo")
    & pruebas_conjuntas_anuales.escala_target.eq("log_real_2024")
    & pruebas_conjuntas_anuales.tipo_covarianza.eq("cluster_hogar")
    & pruebas_conjuntas_anuales.estimador.eq("OLS")
])
display(pd.DataFrame([{
    "ponderados_solicitados": EJECUTAR_MODELOS_PONDERADOS,
    "diseno_disponible_cuatro_anios": DISENO_VALIDO_CUATRO,
    "ponderados_ejecutados": PONDERADOS_EJECUTADOS,
    "normalizacion": "factor dividido por su media dentro de cada año",
}]))

In [ ]:
orden_comparacion = [
    "estimador", "especificacion", "escala_target", "tipo_covarianza", "termino"
]
coeficientes_modelos_anuales = coeficientes_modelos_anuales.sort_values(
    orden_comparacion + ["anio"]
).reset_index(drop=True)
coeficientes_modelos_anuales["signo"] = np.sign(
    coeficientes_modelos_anuales.coeficiente
).astype(int)
coeficientes_modelos_anuales["significativa_descriptiva"] = (
    coeficientes_modelos_anuales.p_value < ALFA
)
grupos = coeficientes_modelos_anuales.groupby(orden_comparacion, dropna=False)
coeficientes_modelos_anuales["signo_previo"] = grupos.signo.shift()
coeficientes_modelos_anuales["significativa_previa"] = grupos.significativa_descriptiva.shift()
coeficientes_modelos_anuales["cambio_signo_descriptivo"] = (
    coeficientes_modelos_anuales.signo_previo.notna()
    & coeficientes_modelos_anuales.signo.ne(coeficientes_modelos_anuales.signo_previo)
)
coeficientes_modelos_anuales["entrada_significancia_descriptiva"] = (
    coeficientes_modelos_anuales.significativa_previa.eq(False)
    & coeficientes_modelos_anuales.significativa_descriptiva
)
coeficientes_modelos_anuales["salida_significancia_descriptiva"] = (
    coeficientes_modelos_anuales.significativa_previa.eq(True)
    & ~coeficientes_modelos_anuales.significativa_descriptiva
)
coeficientes_modelos_anuales["amplitud_intervalo"] = (
    coeficientes_modelos_anuales.intervalo_superior
    - coeficientes_modelos_anuales.intervalo_inferior
)
display(coeficientes_modelos_anuales.loc[
    coeficientes_modelos_anuales.especificacion.eq("regional_nucleo")
    & coeficientes_modelos_anuales.escala_target.eq("log_real_2024")
    & coeficientes_modelos_anuales.tipo_covarianza.eq("cluster_hogar")
    & coeficientes_modelos_anuales.estimador.eq("OLS"),
    ["anio", "variable", "termino", "coeficiente", "error_estandar",
     "intervalo_inferior", "intervalo_superior",
     "significativa_descriptiva", "cambio_signo_descriptivo"]
].head(40))

## 5. Lectura visual de coeficientes anuales

Los forest plots muestran estimacion e IC, no solo estrellas. Las trayectorias unen estimaciones de cortes distintos; la linea **no** representa seguimiento individual. Se separan continuas, binarias, bloques laborales/localidad y geografia. La entidad se grafica en paneles legibles, con diferencias de intercepto respecto de Ciudad de Mexico. Un cambio de signo, o que un IC cruce cero en un año, es descriptivo; la prueba formal vendra de las interacciones.

In [ ]:
tabla_visual = coeficientes_modelos_anuales.loc[
    coeficientes_modelos_anuales.estimador.eq("OLS")
    & coeficientes_modelos_anuales.escala_target.eq("log_real_2024")
    & coeficientes_modelos_anuales.tipo_covarianza.eq("cluster_hogar")
    & coeficientes_modelos_anuales.variable.ne("intercepto")
].copy()
figuras_temporales = {}
colores_anio = {
    2018: "#2070a0", 2020: "#c55a35", 2022: "#488060", 2024: "#65527f"
}


def forest_por_terminos(tabla, terminos, titulo):
    sub = tabla.loc[tabla.termino.isin(terminos)]
    if sub.empty:
        raise ValueError(f"Sin terminos para {titulo}.")
    etiquetas = [t for t in terminos if t in set(sub.termino)]
    fig, ax = plt.subplots(figsize=(10, max(3, 0.55 * len(etiquetas) + 1.5)))
    desplazamientos = {2018: -0.27, 2020: -0.09, 2022: 0.09, 2024: 0.27}
    posiciones = {termino: j for j, termino in enumerate(etiquetas)}
    for anio in ANIOS:
        corte = sub.loc[sub.anio.eq(anio)]
        for _, fila in corte.iterrows():
            y = posiciones[fila.termino] + desplazamientos[anio]
            ax.errorbar(
                fila.coeficiente, y,
                xerr=[[fila.coeficiente - fila.intervalo_inferior],
                      [fila.intervalo_superior - fila.coeficiente]],
                fmt="o", color=colores_anio[anio], capsize=2,
            )
        ax.plot([], [], "o", color=colores_anio[anio], label=str(anio))
    ax.axvline(0, color="0.35", linewidth=0.8)
    ax.set_yticks(range(len(etiquetas)), etiquetas)
    ax.invert_yaxis()
    ax.set_xlabel("Coeficiente log1p real e IC 95% (EE hogar)")
    ax.set_title(titulo)
    ax.legend(ncol=4, frameon=False)
    fig.tight_layout()
    return fig


def trayectorias_por_terminos(tabla, terminos, titulo):
    fig, ejes = plt.subplots(
        len(terminos), 1, figsize=(8, max(3, 2.3 * len(terminos))),
        squeeze=False, sharex=True,
    )
    for ax, termino in zip(ejes[:, 0], terminos):
        corte = tabla.loc[tabla.termino.eq(termino)].sort_values("anio")
        ax.errorbar(
            corte.anio, corte.coeficiente,
            yerr=[corte.coeficiente - corte.intervalo_inferior,
                  corte.intervalo_superior - corte.coeficiente],
            fmt="o-", capsize=3, color="#2070a0",
        )
        ax.axhline(0, color="0.5", linewidth=0.7)
        ax.set_ylabel(termino)
        ax.set_xticks(ANIOS)
    ejes[0, 0].set_title(titulo)
    ejes[-1, 0].set_xlabel("Corte ENIGH (personas diferentes)")
    fig.tight_layout()
    return fig


regional_visual = tabla_visual.loc[
    tabla_visual.especificacion.eq("regional_nucleo")
]
grupos_figura = {
    "numericas": variables_comunes_numericas,
    "binarias": ["sexo_desc", "hablaind_desc", "segsoc_desc"],
    "laborales_categoricas": ["subor_principal_desc", "contrato_principal_desc"],
    "localidad": ["tam_loc_desc"],
    "region": ["region_banxico"],
}
for nombre, variables in grupos_figura.items():
    terminos = regional_visual.loc[
        regional_visual.variable.isin(variables), "termino"
    ].drop_duplicates().tolist()
    fig = forest_por_terminos(regional_visual, terminos, nombre.replace("_", " ").title())
    figuras_temporales[f"forest_{nombre}"] = fig
    display(fig)
for nombre in ["numericas", "binarias"]:
    terminos = regional_visual.loc[
        regional_visual.variable.isin(grupos_figura[nombre]), "termino"
    ].drop_duplicates().tolist()
    fig = trayectorias_por_terminos(
        regional_visual, terminos, f"Trayectorias: {nombre}"
    )
    figuras_temporales[f"trayectorias_{nombre}"] = fig
    display(fig)

if EJECUTAR_ESPECIFICACION_ENTIDAD:
    estatal_visual = tabla_visual.loc[
        tabla_visual.especificacion.eq("estatal_sensibilidad")
        & tabla_visual.variable.eq("entidad")
    ]
    terminos_entidad = estatal_visual.termino.drop_duplicates().tolist()
    for inicio in range(0, len(terminos_entidad), 8):
        bloque = terminos_entidad[inicio:inicio + 8]
        fig = forest_por_terminos(
            estatal_visual, bloque,
            f"Entidades {inicio + 1}-{inicio + len(bloque)} (ref. Ciudad de Mexico)"
        )
        figuras_temporales[f"forest_entidad_{inicio // 8 + 1}"] = fig
        display(fig)

mapa = regional_visual.pivot(index="termino", columns="anio", values="coeficiente")
fig, ax = plt.subplots(figsize=(8, max(5, 0.42 * len(mapa) + 1)))
limite = float(np.nanmax(np.abs(mapa.to_numpy())))
sns.heatmap(mapa, cmap="BrBG", center=0, vmin=-limite, vmax=limite,
            annot=True, fmt=".2f", cbar_kws={"label": "beta log1p real"}, ax=ax)
ax.set_title("Signo y magnitud; no es prueba de cambio temporal")
ax.set_xlabel("Año")
ax.set_ylabel("Termino")
fig.tight_layout()
figuras_temporales["mapa_signos_magnitudes"] = fig
display(fig)

## 6. Modelo pooled con efectos de año

Se apilan los cuatro cortes conservando anio en las llaves. La referencia temporal es 2024; su coeficiente de año vale cero por construccion. Los otros coeficientes de año son diferencias promedio condicionadas al mismo conjunto de predictores, no trayectorias personales. La especificacion regional es principal. Una alternativa estatal reemplaza region, sin agregar entidad y region simultaneamente. La WLS pooled es una sensibilidad: factor dividido por su media dentro de cada año mantiene peso medio uno por corte, de modo que la contribucion entre años sigue aproximadamente sus tamaños muestrales, **no** totales poblacionales calibrados.

In [ ]:
columnas_pool = list(dict.fromkeys(
    KEY_COLS + variables_comunes_numericas + variables_comunes_categoricas
    + variable_geografica_regional + variable_geografica_estatal
    + ["factor", "est_dis", "upm"]
))
partes_pool = []
for anio in ANIOS:
    parte = bases_modelo[anio][columnas_pool].copy()
    nombres = nombres_targets(anio)
    for escala, columna in nombres.items():
        parte[f"target_{escala}"] = pd.to_numeric(
            bases_modelo[anio][columna], errors="coerce"
        ).to_numpy()
    partes_pool.append(parte)
apilada = pd.concat(partes_pool, ignore_index=True)
if len(apilada) != int(auditoria_bases_temporales.n_personas.sum()):
    raise ValueError("El apilado cambio el universo.")
if apilada.duplicated(KEY_COLS).any():
    raise ValueError("Llaves repetidas al apilar años.")
n_hogares_pool = apilada[HOGAR_COLS].drop_duplicates().shape[0]
anios_otros = [anio for anio in ANIOS if anio != ANIO_REFERENCIA]


def matriz_pooled(base, variables):
    X, info = construir_matriz(base, variables)
    for anio in anios_otros:
        termino = f"anio__{anio}"
        X[termino] = base.anio.eq(anio).astype(float)
        info = pd.concat([
            info, pd.DataFrame([{
                "termino": termino, "variable": "anio",
                "categoria": anio, "referencia": ANIO_REFERENCIA,
                "tipo": "efecto_anio",
            }])
        ], ignore_index=True)
    if np.linalg.matrix_rank(X.to_numpy().T @ X.to_numpy()) != X.shape[1]:
        raise ValueError("Pooled sin rango completo.")
    return X, info


filas_coeficientes_pooled = []
resumen_pooled = []
X_pooled_regional, info_pooled_regional = matriz_pooled(
    apilada, ESPECIFICACIONES["regional_nucleo"]
)
y_pool = apilada["target_log_real_2024"].astype(float)
ajuste_pooled_regional = sm.OLS(y_pool, X_pooled_regional).fit()
if MOSTRAR_SUMMARIES_COMPLETOS:
    display(ajuste_pooled_regional.summary())
for covarianza, resultado in variantes_covarianza(
    ajuste_pooled_regional, apilada
).items():
    filas_coeficientes_pooled.append(coeficientes_de(
        resultado, info_pooled_regional,
        {"especificacion": "pooled_regional", "escala_target": "log_real_2024",
         "estimador": "OLS", "tipo_covarianza": covarianza,
         "n_hogares": n_hogares_pool}
    ))
resumen_pooled.append({
    "especificacion": "pooled_regional", "estimador": "OLS",
    "n_personas": len(apilada), "n_hogares": n_hogares_pool,
    "r2": ajuste_pooled_regional.rsquared,
    "r2_ajustado": ajuste_pooled_regional.rsquared_adj,
})

if EJECUTAR_ESPECIFICACION_ENTIDAD:
    X_pooled_estatal, info_pooled_estatal = matriz_pooled(
        apilada, ESPECIFICACIONES["estatal_sensibilidad"]
    )
    ajuste_pooled_estatal = sm.OLS(y_pool, X_pooled_estatal).fit()
    for covarianza, resultado in variantes_covarianza(
        ajuste_pooled_estatal, apilada
    ).items():
        filas_coeficientes_pooled.append(coeficientes_de(
            resultado, info_pooled_estatal,
            {"especificacion": "pooled_estatal_sensibilidad",
             "escala_target": "log_real_2024", "estimador": "OLS",
             "tipo_covarianza": covarianza, "n_hogares": n_hogares_pool}
        ))
    resumen_pooled.append({
        "especificacion": "pooled_estatal_sensibilidad", "estimador": "OLS",
        "n_personas": len(apilada), "n_hogares": n_hogares_pool,
        "r2": ajuste_pooled_estatal.rsquared,
        "r2_ajustado": ajuste_pooled_estatal.rsquared_adj,
    })

if PONDERADOS_EJECUTADOS:
    pesos_pool = pd.to_numeric(apilada.factor, errors="coerce")
    pesos_pool = pesos_pool / pesos_pool.groupby(apilada.anio).transform("mean")
    ajuste_pooled_wls = sm.WLS(
        y_pool, X_pooled_regional, weights=pesos_pool
    ).fit()
    for covarianza, resultado in variantes_covarianza(
        ajuste_pooled_wls, apilada
    ).items():
        filas_coeficientes_pooled.append(coeficientes_de(
            resultado, info_pooled_regional,
            {"especificacion": "pooled_regional",
             "escala_target": "log_real_2024", "estimador": "WLS_sensibilidad",
             "tipo_covarianza": covarianza, "n_hogares": n_hogares_pool}
        ))
    resumen_pooled.append({
        "especificacion": "pooled_regional", "estimador": "WLS_sensibilidad",
        "n_personas": len(apilada), "n_hogares": n_hogares_pool,
        "r2": ajuste_pooled_wls.rsquared,
        "r2_ajustado": ajuste_pooled_wls.rsquared_adj,
    })
coeficientes_modelo_pooled = pd.concat(filas_coeficientes_pooled, ignore_index=True)
display(pd.DataFrame(resumen_pooled))
display(coeficientes_modelo_pooled.loc[
    coeficientes_modelo_pooled.variable.eq("anio")
    & coeficientes_modelo_pooled.tipo_covarianza.eq("cluster_hogar")
    & coeficientes_modelo_pooled.estimador.eq("OLS"),
    ["especificacion", "termino", "coeficiente", "intervalo_inferior",
     "intervalo_superior", "p_value"]
])

## 7. Interacciones con año

El coeficiente principal de un predictor es su asociacion en **2024**. La interaccion predictor × 2018, 2020 o 2022 es la diferencia de coeficiente respecto de 2024. El efecto total en otro año es la suma del coeficiente principal y su interaccion; el IC de esa suma usa la covarianza entre ambos terminos. Para dummies, todo es relativo a la categoria de referencia fija. No se infiere cambio a partir de dos p-values anuales distintos.

Por defecto se incluyen interacciones completas de los bloques individuales, laborales, localidad y region. Para una ejecucion acotada se puede desactivar EJECUTAR_INTERACCIONES_COMPLETAS y usar BLOQUES_INTERACCION y/o VARIABLES_INTERACCION_MANUAL. Escolaridad solo entrara tras homologacion aprobada. La alternativa estatal no incluye automaticamente pendientes diferentes por estado.

In [ ]:
if EJECUTAR_INTERACCIONES_COMPLETAS:
    variables_interaccion = (
        BLOQUES["individuales"] + BLOQUES["laborales"]
        + BLOQUES["localidad"] + BLOQUES["region"]
    )
else:
    bloques_invalidos = sorted(set(BLOQUES_INTERACCION) - set(BLOQUES))
    if bloques_invalidos:
        raise ValueError(f"Bloques de interaccion desconocidos: {bloques_invalidos}")
    variables_interaccion = list(dict.fromkeys(
        [v for bloque in BLOQUES_INTERACCION for v in BLOQUES[bloque]]
        + VARIABLES_INTERACCION_MANUAL
    ))
variables_interaccion = list(dict.fromkeys(variables_interaccion))
if not variables_interaccion or not set(variables_interaccion).issubset(
    ESPECIFICACIONES["regional_nucleo"]
):
    raise ValueError("Interacciones vacias o fuera de la especificacion regional.")
display(pd.DataFrame([
    {"bloque": bloque, "variables_interactuadas": " | ".join(
        v for v in lista if v in variables_interaccion
    ) or "no incluidas"}
    for bloque, lista in BLOQUES.items()
]))

X_interacciones = X_pooled_regional.copy()
info_interacciones = info_pooled_regional.copy()
filas_interacciones = []
info_base = info_pooled_regional.loc[
    info_pooled_regional.variable.isin(variables_interaccion)
    & ~info_pooled_regional.tipo.eq("efecto_anio")
].copy()
for variable in variables_interaccion:
    terminos_base = info_base.loc[info_base.variable.eq(variable)]
    if terminos_base.empty:
        raise ValueError(f"Sin terminos base para {variable}.")
    for _, detalle in terminos_base.iterrows():
        for anio in anios_otros:
            termino = f"{detalle.termino}:anio__{anio}"
            X_interacciones[termino] = (
                X_pooled_regional[detalle.termino]
                * X_pooled_regional[f"anio__{anio}"]
            )
            registro = {
                "termino": termino, "variable": variable,
                "categoria": detalle.categoria,
                "referencia": detalle.referencia, "tipo": "interaccion",
                "termino_base": detalle.termino, "anio": anio,
            }
            filas_interacciones.append(registro)
            info_interacciones = pd.concat([
                info_interacciones,
                pd.DataFrame([{k: registro[k] for k in [
                    "termino", "variable", "categoria", "referencia", "tipo"
                ]}])
            ], ignore_index=True)
interacciones_en_memoria = pd.DataFrame(filas_interacciones)
if np.linalg.matrix_rank(
    X_interacciones.to_numpy().T @ X_interacciones.to_numpy()
) != X_interacciones.shape[1]:
    raise ValueError("Interacciones sin rango completo; revisar categorias.")
ajuste_interacciones = sm.OLS(y_pool, X_interacciones).fit()
modelo_interacciones_cluster = variantes_covarianza(
    ajuste_interacciones, apilada
)["cluster_hogar"]
if MOSTRAR_SUMMARIES_COMPLETOS:
    display(ajuste_interacciones.summary())
coeficientes_modelo_pooled = pd.concat([
    coeficientes_modelo_pooled,
    coeficientes_de(
        modelo_interacciones_cluster, info_interacciones,
        {"especificacion": "pooled_interacciones_regionales",
         "escala_target": "log_real_2024", "estimador": "OLS",
         "tipo_covarianza": "cluster_hogar", "n_hogares": n_hogares_pool}
    )
], ignore_index=True)
display(interacciones_en_memoria[
    ["variable", "termino_base", "anio", "termino"]
].head(20))

## 8. Efectos totales y pruebas de Wald

Para cada termino interactuado se calculan efecto total en cada corte y contraste contra 2024, con IC de la combinacion lineal. Las pruebas Wald contrastan **conjuntos de interacciones iguales a cero**, por predictor continuo, por bloque categorico, por familias laboral/individual/territorial y por el conjunto total. P pequeño indica evidencia estadistica de diferencia condicionada a esta especificacion; no implica relevancia sustantiva ni causalidad. Con muchas pruebas, la correccion por multiplicidad sigue pendiente de decision metodologica.

In [ ]:
nombres_inter = modelo_interacciones_cluster.model.exog_names
parametros_inter = np.asarray(modelo_interacciones_cluster.params)
cov_inter = np.asarray(modelo_interacciones_cluster.cov_params())
zcrit = norm.ppf(1 - ALFA / 2)


def contraste_lineal(pesos):
    beta = float(pesos @ parametros_inter)
    varianza = float(pesos @ cov_inter @ pesos)
    if varianza < -1e-10:
        raise ValueError("Varianza negativa del contraste.")
    ee = float(np.sqrt(max(varianza, 0)))
    z = beta / ee if ee > 0 else np.nan
    return {
        "coeficiente": beta, "error_estandar": ee,
        "intervalo_inferior": beta - zcrit * ee,
        "intervalo_superior": beta + zcrit * ee,
        "p_value": float(2 * norm.sf(abs(z))) if np.isfinite(z) else np.nan,
    }


filas_efectos = []
for termino_base, bloque in interacciones_en_memoria.groupby("termino_base", sort=False):
    detalle = bloque.iloc[0]
    for anio in ANIOS:
        pesos_total = np.zeros(len(nombres_inter))
        pesos_total[nombres_inter.index(termino_base)] = 1
        pesos_diferencia = np.zeros(len(nombres_inter))
        if anio != ANIO_REFERENCIA:
            termino = bloque.loc[bloque.anio.eq(anio), "termino"].iloc[0]
            pesos_total[nombres_inter.index(termino)] = 1
            pesos_diferencia[nombres_inter.index(termino)] = 1
        total = contraste_lineal(pesos_total)
        diferencia = (
            contraste_lineal(pesos_diferencia) if anio != ANIO_REFERENCIA
            else {"coeficiente": 0.0, "error_estandar": 0.0,
                  "intervalo_inferior": 0.0, "intervalo_superior": 0.0,
                  "p_value": np.nan}
        )
        filas_efectos.append({
            "anio": anio, "anio_referencia": ANIO_REFERENCIA,
            "variable": detalle.variable, "termino_base": termino_base,
            "categoria": detalle.categoria, "referencia": detalle.referencia,
            **{f"efecto_total_{k}": v for k, v in total.items()},
            **{f"diferencia_vs_2024_{k}": v for k, v in diferencia.items()},
            "tipo_covarianza": "cluster_hogar",
            "n_personas": len(apilada), "n_hogares": n_hogares_pool,
        })
efectos_totales_interacciones = pd.DataFrame(filas_efectos)
display(efectos_totales_interacciones.head(20))


def prueba_wald_temporal(nombre, variables, tipo):
    terminos = interacciones_en_memoria.loc[
        interacciones_en_memoria.variable.isin(variables), "termino"
    ].tolist()
    if not terminos:
        return {
            "hipotesis": nombre, "tipo": tipo, "variables": " | ".join(variables),
            "terminos_probados": "", "estadistico": np.nan,
            "gl_restricciones": 0, "gl_denominador": np.nan, "p_value": np.nan,
            "interpretacion": "No incluido en la ejecucion parcial.",
            "conclusion_descriptiva": "no_estimado",
        }
    R = np.zeros((len(terminos), len(nombres_inter)))
    for j, termino in enumerate(terminos):
        R[j, nombres_inter.index(termino)] = 1
    prueba = modelo_interacciones_cluster.wald_test(
        R, use_f=False, scalar=True
    )
    p = float(prueba.pvalue)
    return {
        "hipotesis": nombre, "tipo": tipo,
        "variables": " | ".join(variables),
        "terminos_probados": " | ".join(terminos),
        "estadistico": float(prueba.statistic),
        "gl_restricciones": len(terminos),
        "gl_denominador": np.nan, "p_value": p,
        "interpretacion": "H0: interacciones de estos terminos iguales a cero.",
        "conclusion_descriptiva": (
            "evidencia de diferencia temporal condicionada, no causal"
            if p < ALFA else
            "sin evidencia suficiente de diferencia temporal condicionada"
        ),
    }


filas_wald = []
for variable in variables_comunes_numericas:
    filas_wald.append(
        prueba_wald_temporal(variable, [variable], "continua_por_anio")
    )
for variable in variables_comunes_categoricas + ["region_banxico"]:
    filas_wald.append(
        prueba_wald_temporal(variable, [variable], "bloque_categorico_por_anio")
    )
for bloque, variables in {
    "individuales": BLOQUES["individuales"],
    "laborales": BLOQUES["laborales"],
    "territoriales": BLOQUES["localidad"] + BLOQUES["region"],
    "todas_las_interacciones": variables_interaccion,
}.items():
    filas_wald.append(
        prueba_wald_temporal(bloque, variables, "familia_temporal")
    )
pruebas_wald_temporales = pd.DataFrame(filas_wald)
display(pruebas_wald_temporales[
    ["hipotesis", "tipo", "gl_restricciones", "estadistico", "p_value",
     "conclusion_descriptiva"]
])

## 9. Geografia, ponderacion y antecedente 2024

Los coeficientes regionales/estatales comparan el **intercepto condicionado** con Centro/Ciudad de Mexico, respectivamente; no dicen que todas las pendientes cambien por territorio. Para estudiar pendientes heterogeneas por estado haria falta una fase posterior, inicialmente con interacciones regionales justificadas. En este notebook las interacciones territoriales son con **año**, no con todos los predictores de cada entidad.

Factor, est_dis y upm se auditan. La WLS normaliza factor a media uno dentro de cada corte, tanto en modelos anuales como pooled; asi se controla la escala numerica y se evita que la suma de factores de un año domine automaticamente el pooled. La sensibilidad sigue sin tratar formalmente estratificacion, UPM y replicas del diseño ENIGH. Agrupar EE por hogar atiende dependencia intrahogar, no sustituye dicho diseño.

La lista historica de variables significativas de 2024 es solo un **antecedente descriptivo** del notebook 12/13 y no filtra el nucleo. La tabla siguiente reconstruye significancia por termino de la **nueva especificacion** cuando se ejecute, por lo que no debe confundirse con los coeficientes de entrenamiento del notebook 13.

In [ ]:
comparacion_geografica = coeficientes_modelos_anuales.loc[
    coeficientes_modelos_anuales.estimador.eq("OLS")
    & coeficientes_modelos_anuales.escala_target.eq("log_real_2024")
    & coeficientes_modelos_anuales.tipo_covarianza.eq("cluster_hogar")
    & coeficientes_modelos_anuales.variable.isin(["region_banxico", "entidad"])
].copy()
comparacion_geografica["origen_modelo"] = "anual"
geo_pooled = coeficientes_modelo_pooled.loc[
    coeficientes_modelo_pooled.estimador.eq("OLS")
    & coeficientes_modelo_pooled.tipo_covarianza.eq("cluster_hogar")
    & coeficientes_modelo_pooled.variable.isin(["region_banxico", "entidad"])
    & ~coeficientes_modelo_pooled.especificacion.eq(
        "pooled_interacciones_regionales"
    )
].copy()
geo_pooled["anio"] = pd.NA
geo_pooled["origen_modelo"] = "pooled"
comparacion_geografica = pd.concat(
    [comparacion_geografica, geo_pooled], ignore_index=True
)
comparacion_geografica["interpretacion"] = (
    "diferencia de intercepto condicionado vs referencia geografica; "
    "no implica cambio de pendientes ni causalidad"
)
display(comparacion_geografica[
    ["origen_modelo", "anio", "especificacion", "variable", "categoria",
     "referencia", "coeficiente", "intervalo_inferior", "intervalo_superior"]
].head(40))

ANTECEDENTE_2024_VARIABLES = [
    "edad", "n_trabajos", "horas_trabajos_total", "sexo_desc",
    "nivelaprob_desc", "parentesco_desc", "hablaind_desc",
    "segsoc_desc", "subor_principal_desc", "contrato_principal_desc",
    "region_banxico", "tam_loc_desc",
]
principal_anual = coeficientes_modelos_anuales.loc[
    coeficientes_modelos_anuales.estimador.eq("OLS")
    & coeficientes_modelos_anuales.especificacion.eq("regional_nucleo")
    & coeficientes_modelos_anuales.escala_target.eq("log_real_2024")
    & coeficientes_modelos_anuales.tipo_covarianza.eq("cluster_hogar")
    & coeficientes_modelos_anuales.variable.ne("intercepto")
].copy()
wald_por_variable = pruebas_wald_temporales.loc[
    pruebas_wald_temporales.tipo.isin(
        ["continua_por_anio", "bloque_categorico_por_anio"]
    )
].set_index("hipotesis")
prueba_2024 = pruebas_conjuntas_anuales.loc[
    pruebas_conjuntas_anuales.anio.eq(2024)
    & pruebas_conjuntas_anuales.estimador.eq("OLS")
    & pruebas_conjuntas_anuales.especificacion.eq("regional_nucleo")
    & pruebas_conjuntas_anuales.escala_target.eq("log_real_2024")
    & pruebas_conjuntas_anuales.tipo_covarianza.eq("cluster_hogar")
].set_index("variable")
filas_antecedentes = []
for _, fila in principal_anual.loc[principal_anual.anio.eq(2024)].iterrows():
    otros = principal_anual.loc[
        principal_anual.termino.eq(fila.termino)
        & principal_anual.anio.ne(2024)
    ].sort_values("anio")
    p_wald = (
        float(wald_por_variable.loc[fila.variable, "p_value"])
        if fila.variable in wald_por_variable.index else np.nan
    )
    p_bloque = (
        float(prueba_2024.loc[fila.variable, "p_value"])
        if fila.variable in prueba_2024.index else np.nan
    )
    filas_antecedentes.append({
        "termino": fila.termino, "variable": fila.variable,
        "en_antecedente_exploratorio_2024":
            fila.variable in ANTECEDENTE_2024_VARIABLES,
        "significativa_2024_nucleo_actual":
            bool(fila.significativa_descriptiva),
        "signo_2024": int(fila.signo),
        "otros_anios_descriptivo": " | ".join(
            f"{int(r.anio)}: signo {int(r.signo)}, "
            f"p<{ALFA}: {bool(r.significativa_descriptiva)}"
            for _, r in otros.iterrows()
        ),
        "p_wald_temporal_variable": p_wald,
        "interaccion_temporal_significativa":
            bool(p_wald < ALFA) if np.isfinite(p_wald) else None,
        "p_prueba_conjunta_2024": p_bloque,
        "conclusion": (
            "diferencia temporal condicionada; no causal"
            if np.isfinite(p_wald) and p_wald < ALFA else
            "cambio de p anual no demuestra diferencia temporal"
        ),
    })
for variable in ["nivelaprob_desc", "parentesco_desc"]:
    filas_antecedentes.append({
        "termino": None, "variable": variable,
        "en_antecedente_exploratorio_2024": True,
        "significativa_2024_nucleo_actual": None,
        "signo_2024": None, "otros_anios_descriptivo": None,
        "p_wald_temporal_variable": np.nan,
        "interaccion_temporal_significativa": None,
        "p_prueba_conjunta_2024": np.nan,
        "conclusion": "pendiente de armonizacion; fuera del nucleo",
    })
antecedentes_2024 = pd.DataFrame(filas_antecedentes)
display(antecedentes_2024)

## 10. Guia de interpretacion y limites

- **Significativa en ambos años, betas diferentes:** las dos asociaciones pueden ser distintas; solo el contraste de interaccion y su IC prueban formalmente esa diferencia.
- **Significativa solo en uno:** la diferencia entre estimaciones puede seguir siendo compatible con cero. No se concluye cambio a partir de los dos p-values aislados.
- **Dummies no significativas por separado, bloque significativo:** el test conjunto evalua todas las categorias a la vez y responde otra pregunta.
- **Magnitud sustantiva:** un beta pequeno con p muy bajo en muestras grandes puede tener poca importancia practica. Para log1p, interpretar 100(exp(beta)-1) sobre 1+ingreso, junto con el IC y la unidad de cada predictor.
- **Nivel monetario:** no equiparar pesos nominales de años distintos. La conversion real es aproximada y pendiente de aprobacion.
- **Alcance:** solo personas adultas con ingreso laboral/de negocio positivo; sin seguimiento longitudinal, ajuste completo por diseño muestral ni interpretacion causal. Wald prueba restricciones del modelo especificado, no mecanismos economicos.
- **Pendiente:** correccion por pruebas multiples, homologacion educativa, pertinencia del factor real individual y eventual inferencia con estratos/UPM y replicas.

## 11. Exportacion al final

Todas las tablas y figuras siguientes provienen de objetos creados arriba durante **esta misma ejecucion manual**. No se leen coeficientes ni graficas anteriores y no se recalculan modelos en esta celda. El manifest conserva el estado metodologico pendiente_ejecucion_manual hasta revisar y aprobar resultados; no debe confundirse con una corrida verificada por Codex.

In [ ]:
tablas_salida = {
    "auditoria_bases_temporales": auditoria_bases_temporales,
    "comparabilidad_variables": comparabilidad_variables,
    "referencias_categoricas": referencias_categoricas,
    "metricas_modelos_anuales": metricas_modelos_anuales,
    "coeficientes_modelos_anuales": coeficientes_modelos_anuales,
    "pruebas_conjuntas_anuales": pruebas_conjuntas_anuales,
    "coeficientes_modelo_pooled": coeficientes_modelo_pooled,
    "efectos_totales_interacciones": efectos_totales_interacciones,
    "pruebas_wald_temporales": pruebas_wald_temporales,
    "comparacion_geografica": comparacion_geografica,
    "antecedentes_2024": antecedentes_2024,
}
manifest_comparacion_temporal = {
    "estado": "pendiente_ejecucion_manual",
    "nota_estado": "Notebook preparado; resultados requieren ejecucion y revision humana.",
    "unidad_analisis": "persona",
    "universo": "edad>=18 e ingreso laboral/de negocio individual nominal trimestral>0",
    "anios": ANIOS,
    "anio_referencia": ANIO_REFERENCIA,
    "archivos_entrada": {
        str(anio): str(ruta.relative_to(PROJECT_ROOT))
        for anio, ruta in INPUTS.items()
    },
    "variables_comunes_numericas": variables_comunes_numericas,
    "variables_comunes_categoricas": variables_comunes_categoricas,
    "geografia_principal": variable_geografica_regional,
    "geografia_sensibilidad": variable_geografica_estatal
                              if EJECUTAR_ESPECIFICACION_ENTIDAD else [],
    "variables_ampliadas": [],
    "estado_escolaridad": estado_escolaridad,
    "variables_pendientes": variables_pendientes,
    "referencias": REFERENCIAS,
    "categorias_comunes": CATEGORIAS_COMUNES,
    "targets": {
        str(anio): nombres_targets(anio) for anio in ANIOS
    },
    "target_modelo_principal": "log_real_2024",
    "target_modelo_secundario": "nivel_nominal",
    "estimadores": ["OLS"] + (
        ["WLS_sensibilidad"] if PONDERADOS_EJECUTADOS else []
    ),
    "covarianzas_anuales": ["convencional", "HC3", "cluster_hogar"],
    "covarianza_interacciones": "cluster_hogar",
    "interacciones": variables_interaccion,
    "interacciones_completas": EJECUTAR_INTERACCIONES_COMPLETAS,
    "ponderacion_sensibilidad_ejecutada": PONDERADOS_EJECUTADOS,
    "normalizacion_factor": "factor/media(factor) dentro de cada año",
    "diseno": "factor, est_dis, upm disponibles; WLS+EE hogar no es diseño complejo completo",
    "advertencia_monetaria": (
        "Factor anual calibrado al benchmark de ingreso corriente del hogar; "
        "uso en ingreso laboral individual pendiente de aprobacion metodologica."
    ),
    "metodo": "asociaciones condicionadas en cortes transversales repetidos, no causales",
    "versiones": {
        "pandas": pd.__version__, "numpy": np.__version__,
        "statsmodels": sm.__version__,
    },
}
if EXPORTAR_RESULTADOS:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    for nombre, tabla in tablas_salida.items():
        tabla.to_csv(OUTPUT_DIR / f"{nombre}.csv", index=False)
    figuras_dir = OUTPUT_DIR / "figures"
    figuras_dir.mkdir(parents=True, exist_ok=True)
    for nombre, figura in figuras_temporales.items():
        figura.savefig(figuras_dir / f"{nombre}.png", dpi=150,
                       bbox_inches="tight")
    (OUTPUT_DIR / "manifest_comparacion_temporal.json").write_text(
        json.dumps(manifest_comparacion_temporal, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )
display(pd.DataFrame([{
    "exportacion": "preparada" if EXPORTAR_RESULTADOS else "desactivada",
    "ruta": str(OUTPUT_DIR),
    "tablas": len(tablas_salida), "figuras": len(figuras_temporales),
    "estado_metodologico": manifest_comparacion_temporal["estado"],
}]))